In [1]:
# !pip install -U datasets
# !pip install evaluate
# !pip install rouge_score
# conda install numpy pandas tqdm nltk jupyter
# pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu126
# pip install transformers datasets accelerate evaluate sentencepiece
# pip install "protobuf>=3.19.6,<6" --upgrade

In [1]:
import os
import numpy as np
import pandas as pd
from tqdm.auto import tqdm

from transformers import AutoTokenizer, MT5ForConditionalGeneration, AutoModelForSeq2SeqLM
from transformers import DataCollatorForSeq2Seq, get_scheduler, TrainingArguments, Trainer
from transformers import Seq2SeqTrainingArguments, Seq2SeqTrainer
from datasets import load_dataset,load_from_disk, DatasetDict

from torch.utils.data import DataLoader
from torch.optim import AdamW
import torch

from accelerate import Accelerator
import evaluate
import nltk
from nltk.tokenize import sent_tokenize

import wandb

In [2]:
import torch

print("PyTorch compiled with CUDA:", torch.version.cuda)
print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("CUDA version (compiled):", torch.version.cuda)
print("cuDNN version:", torch.backends.cudnn.version())
print("GPU name:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "No GPU")


PyTorch compiled with CUDA: 12.6
PyTorch version: 2.7.1+cu126
CUDA available: True
CUDA version (compiled): 12.6
cuDNN version: 90501
GPU name: NVIDIA A100-SXM4-40GB


In [3]:
# from huggingface_hub import login
# from google.colab import userdata

# login(userdata.get('HF_TOKEN'))

In [4]:
device = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")
dataset = load_dataset('/project/lt200246-mmacma/Big_seq2seq/data/text-to-gloss_ver1', cache_dir=None)

In [5]:
dataset

DatasetDict({
    train: Dataset({
        features: ['no', 'text', 'gloss_sequence'],
        num_rows: 1397
    })
    eval: Dataset({
        features: ['no', 'text', 'gloss_sequence'],
        num_rows: 200
    })
    test: Dataset({
        features: ['no', 'text', 'gloss_sequence'],
        num_rows: 50
    })
})

In [6]:
dataset['train'][0]

{'no': 186,
 'text': 'ภาคใต้ ช่วงนี้ฝนเริ่มลดลงนะคะ มีฝนร้อยละ 30-40 ของพื้นที่ แต่ว่าส่วนมากก็ยังคงตกได้อยู่บริเวณฝั่งอ่าวไทยค่ะ',
 'gloss_sequence': 'ภาคใต้|*วันนี้(/ช่วงนี้)|#c ฝนตก + ลดน้อยลง + ในหลายพื้นที่[มีทิศทางประกอบตั้งแต่ฝนตก การขยายท่ามือ "ลดน้อยลง" โดยใช้สีหน้า เป็นตัวเชื่อมกับร้อยละของฝนในแต่ละพื้นที่ ใช้ทิศทางการเคลื่อนไหวรอบ ๆ]|*เปอร์เซ็นต์(/ร้อยละ)|30|ถึง|40|*กับ(/ที่)|*ฝนตกหนัก[ใช้สีหน้าถึงความหนักของปริมาณฝน]|*ทะเล(/คลื่น)|#d อ่าวไทย[ฝ่ามือหันนิ้วโป้งเข้าตัว มือปัดฝั่งนิ้วก้อย]|#c ฝนตก + ในหลายพื้นที่[มีทิศทางประกอบตั้งแต่ฝนตก ใช้สีหน้า "ปานกลาง" เป็นตัวเชื่อมกับร้อยละของฝนในแต่ละพื้นที่ ใช้ทิศทางการเคลื่อนไหวรอบ ๆ]'}

# Model MT5

In [7]:
model_path = '/project/lt200246-mmacma/Big_seq2seq/model/google/mt5-xl'
model = MT5ForConditionalGeneration.from_pretrained(model_path, device_map='auto')
tokenizer = AutoTokenizer.from_pretrained(model_path)

/lustrefs/disk/project/lt200246-mmacma/Big_seq2seq/mt5/env_mt5/lib/python3.10/site-packages/accelerate/utils/modeling.py:1614: UserWarning: The following device_map keys do not match any submodules in the model: ['decoder.embed_tokens', 'encoder.embed_tokens']
  warnings.warn(
You are using the default legacy behaviour of the <class 'transformers.models.t5.tokenization_t5.T5Tokenizer'>. This is expected, and simply means that the `legacy` (previous) behavior will be used so nothing changes for you. If you want to use the new behaviour, set `legacy=False`. This should only be set if you understand what it means, and thoroughly read the reason why this was added as explained in https://github.com/huggingface/transformers/pull/24565
/lustrefs/disk/project/lt200246-mmacma/Big_seq2seq/mt5/env_mt5/lib/python3.10/site-packages/transformers/convert_slow_tokenizer.py:564: UserWarning: The sentencepiece tokenizer that you are converting to a fast tokenizer uses the byte fallback option which is 

In [8]:
model.device

device(type='cuda', index=1)

In [9]:
print(model.config)
print(model.generation_config)

MT5Config {
  "architectures": [
    "MT5ForConditionalGeneration"
  ],
  "classifier_dropout": 0.0,
  "d_ff": 5120,
  "d_kv": 64,
  "d_model": 2048,
  "decoder_start_token_id": 0,
  "dense_act_fn": "gelu_new",
  "dropout_rate": 0.1,
  "eos_token_id": 1,
  "feed_forward_proj": "gated-gelu",
  "initializer_factor": 1.0,
  "is_encoder_decoder": true,
  "is_gated_act": true,
  "layer_norm_epsilon": 1e-06,
  "model_type": "mt5",
  "num_decoder_layers": 24,
  "num_heads": 32,
  "num_layers": 24,
  "output_past": true,
  "pad_token_id": 0,
  "relative_attention_max_distance": 128,
  "relative_attention_num_buckets": 32,
  "tie_word_embeddings": false,
  "tokenizer_class": "T5Tokenizer",
  "torch_dtype": "float32",
  "transformers_version": "4.53.2",
  "use_cache": true,
  "vocab_size": 250112
}

GenerationConfig {
  "decoder_start_token_id": 0,
  "eos_token_id": 1,
  "pad_token_id": 0
}



In [8]:
#Test
input = tokenizer("และปิดท้ายกันที่กรุงเทพมหานครและปริมณฑล อุณหภูมิต่ำสุด 24 องศา สูงสุด 35 องศา มีฝนฟ้าคะนองร้อยละ 70 ของพื้นที่ค่ะ")
output = tokenizer("#c กรุงเทพมหานคร + จังหวัด + พื้นที่ใกล้เคียง(กรุงเทพมหานครปริมณฑล)|*วันนี้(/ช่วงนี้)|เย็น|อุณหภูมิต่ำ|#c 20+4(24)|ร้อน|อุณหภูมิสูง|แตะถึง|#c 30+5(35)|#c ฝนตก + ในหลายพื้นที่[มีทิศทางประกอบตั้งแต่ฝนตก ใช้สีหน้า 'ปานกลาง' เป็นตัวเชื่อมกับร้อยละของฝนในแต่ละพื้นที่ ใช้ทิศทางการเคลื่อนไหวรอบ ๆ]|*เปอร์เซ็นต์(/ร้อยละ)|70")
print("input part")
print(input)
print(tokenizer.convert_ids_to_tokens(input.input_ids))
print(tokenizer.decode(input.input_ids))
print("-----------------------------------------")
print("output part")
print(output)
print(tokenizer.convert_ids_to_tokens(output.input_ids))
print(tokenizer.decode(output.input_ids))

input part
{'input_ids': [3324, 50405, 120305, 7428, 1549, 111284, 2091, 165859, 232626, 259, 180142, 43750, 32921, 840, 259, 210531, 259, 74484, 2307, 259, 210531, 13596, 151422, 87678, 78618, 226150, 200487, 2595, 259, 1881, 42576, 22416, 1], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]}
['▁และ', 'ปิด', 'ท้าย', 'กัน', 'ที่', 'กรุงเทพมหานคร', 'และ', 'ปริ', 'มณฑล', '▁', 'อุณหภูมิ', 'ต่ํา', 'สุด', '▁24', '▁', 'องศา', '▁', 'สูงสุด', '▁35', '▁', 'องศา', '▁มี', 'ฝน', 'ฟ้า', 'คะ', 'นอง', 'ร้อยละ', '▁70', '▁', 'ของ', 'พื้นที่', 'ค่ะ', '</s>']
และปิดท้ายกันที่กรุงเทพมหานครและปริมณฑล อุณหภูมิต่ําสุด 24 องศา สูงสุด 35 องศา มีฝนฟ้าคะนองร้อยละ 70 ของพื้นที่ค่ะ</s>
-----------------------------------------
output part
{'input_ids': [387, 297, 259, 111284, 744, 97550, 744, 259, 42576, 158650, 312, 111284, 165859, 232626, 46198, 894, 43251, 27366, 215735, 46198, 73313, 409, 180142, 43750, 409, 717, 297, 628, 51351, 181343, 409,

## Tokenizer

### Preprocess data

In [8]:
lens = [len(tokenizer.encode(g)) for g in dataset["train"]["text"]]
print(f"Thai Min: {min(lens)}, Avg: {sum(lens)/len(lens):.2f}, Max: {max(lens)}")

lens = [len(tokenizer.encode(g)) for g in dataset["train"]["gloss_sequence"]]
print(f"Sign Min: {min(lens)}, Avg: {sum(lens)/len(lens):.2f}, Max: {max(lens)}")

Thai Min: 4, Avg: 34.64, Max: 96
Sign Min: 6, Avg: 86.36, Max: 281


In [9]:
max_input_length = 200
max_target_length = 300

def preprocess_function(examples):
    
    target_input = f'thai to gloss: {examples["text"]}'
    model_inputs = tokenizer(target_input,
                             text_target=examples["gloss_sequence"],
                             max_length=max_input_length,
                             truncation=True)

    return model_inputs

x = preprocess_function(dataset['train'][0])
print(x)
print(tokenizer.decode(x['input_ids']))
print(tokenizer.decode(x['labels']))

{'input_ids': [15628, 288, 259, 61880, 267, 259, 30931, 89051, 259, 215735, 151422, 54434, 135963, 66998, 13596, 151422, 200487, 259, 110969, 259, 1881, 42576, 259, 163102, 233644, 8302, 102912, 81149, 2752, 11103, 72364, 132990, 11740, 181531, 11984, 22416, 1], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], 'labels': [259, 30931, 89051, 409, 894, 43251, 27366, 215735, 46198, 717, 297, 259, 151422, 81149, 744, 99099, 225994, 744, 21637, 34362, 42576, 765, 5490, 203448, 89559, 50472, 151422, 81149, 12593, 96422, 93229, 28017, 313, 43790, 225994, 311, 259, 141126, 24040, 11922, 259, 185837, 192063, 4395, 200487, 1881, 151422, 160435, 42576, 116649, 203448, 3755, 224251, 42947, 259, 4571, 439, 409, 894, 180411, 149559, 41071, 27366, 200487, 46198, 1249, 409, 16485, 409, 2334, 409, 894, 4395, 27366, 1549, 46198, 894, 151422, 81149, 88618, 765, 12066, 24040, 11922, 212730, 88618, 1881, 127515, 151422, 439, 4

In [10]:
tokenized_dataset = dataset.map(preprocess_function)

Map:   0%|          | 0/1397 [00:00<?, ? examples/s]

Map:   0%|          | 0/200 [00:00<?, ? examples/s]

Map:   0%|          | 0/50 [00:00<?, ? examples/s]

In [11]:
tokenized_dataset

DatasetDict({
    train: Dataset({
        features: ['no', 'text', 'gloss_sequence', 'input_ids', 'attention_mask', 'labels'],
        num_rows: 1397
    })
    eval: Dataset({
        features: ['no', 'text', 'gloss_sequence', 'input_ids', 'attention_mask', 'labels'],
        num_rows: 200
    })
    test: Dataset({
        features: ['no', 'text', 'gloss_sequence', 'input_ids', 'attention_mask', 'labels'],
        num_rows: 50
    })
})

### Data collator

In [12]:
data_collator = DataCollatorForSeq2Seq(tokenizer=tokenizer, model=model, padding="longest", return_tensors='pt')

In [13]:
tokenized_dataset = tokenized_dataset.remove_columns(dataset['train'].column_names)
tokenized_dataset

DatasetDict({
    train: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 1397
    })
    eval: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 200
    })
    test: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 50
    })
})

In [14]:
features = [tokenized_dataset['train'][i] for i in range(2)]
data_collator(features)

{'input_ids': tensor([[ 15628,    288,    259,  61880,    267,    259,  30931,  89051,    259,
         215735, 151422,  54434, 135963,  66998,  13596, 151422, 200487,    259,
         110969,    259,   1881,  42576,    259, 163102, 233644,   8302, 102912,
          81149,   2752,  11103,  72364, 132990,  11740, 181531,  11984,  22416,
              1,      0,      0,      0,      0,      0,      0],
        [ 15628,    288,    259,  61880,    267,    259,  30931, 238655,   5490,
          75346,  73313,   2361,  61896, 119793,    259, 180142, 227125,  15321,
            259, 210531,  13596, 151422, 174801,  31879,  34797,    259, 233644,
         151422,   6494,  81149,   9542,  61896, 158534,   1881,  30931,    259,
         180142,  43750,  32921,    963,    259, 210531,      1]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0],
        [1, 1, 1, 1, 1, 1, 1, 1, 1, 1,

### Data loader

In [15]:
batch_size = 8

test_dataloader = DataLoader(tokenized_dataset["test"],
                             collate_fn=data_collator,
                             batch_size=batch_size)

In [19]:
#Optimize
optimizer = AdamW(model.parameters(), lr=2e-5)

wandb.init(
    project="seq2seq-training-dataver5",
    name="mt5-xl",
    mode="offline" 
)

## Train Model

In [20]:
from transformers import Seq2SeqTrainer, Seq2SeqTrainingArguments, DataCollatorForSeq2Seq

In [21]:
training_args = Seq2SeqTrainingArguments(
    output_dir="/project/lt200246-mmacma/Big_seq2seq/trained_model/model_use_data5/mt5",
    save_strategy= "no",
    num_train_epochs=8,
    do_train=True,
    per_device_train_batch_size=4,
    gradient_checkpointing=True,
    gradient_accumulation_steps=8,

    learning_rate=5e-05,
    lr_scheduler_type="linear",

    do_eval=True,
    eval_strategy="epoch",
    per_device_eval_batch_size=4,

    logging_strategy="epoch",
    report_to="wandb"
)

In [22]:
trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["eval"],
    processing_class=tokenizer,
    data_collator=data_collator,
)

In [23]:
trainer.train()

wandb: WARNING The `run_name` is currently set to the same value as `TrainingArguments.output_dir`. If this was not intended, please specify a different run name by setting the `TrainingArguments.run_name` parameter.
`use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`...


Epoch,Training Loss,Validation Loss
1,1.657000,0.676399
2,0.657000,0.570697
3,0.557000,0.541755
4,0.502000,0.527369
5,0.467200,0.526068
6,0.437500,0.525822
7,0.415200,0.525735
8,0.400000,0.526165


Passing a tuple of `past_key_values` is deprecated and will be removed in Transformers v4.48.0. You should pass an instance of `EncoderDecoderCache` instead, e.g. `past_key_values=EncoderDecoderCache.from_legacy_cache(past_key_values)`.


TrainOutput(global_step=776, training_loss=0.6365973875694668, metrics={'train_runtime': 3430.7352, 'train_samples_per_second': 7.196, 'train_steps_per_second': 0.226, 'total_flos': 2.519005050863616e+16, 'train_loss': 0.6365973875694668, 'epoch': 8.0})

In [24]:
trainer.save_model()

# Inference

In [7]:
model = AutoModelForSeq2SeqLM.from_pretrained("/project/lt200246-mmacma/Big_seq2seq/trained_model/model_use_data1/mt5", device_map='auto')
tokenizer = AutoTokenizer.from_pretrained("/project/lt200246-mmacma/Big_seq2seq/trained_model/model_use_data1/mt5")

Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

/lustrefs/disk/project/lt200246-mmacma/Big_seq2seq/mt5/env_mt5/lib/python3.10/site-packages/accelerate/utils/modeling.py:1614: UserWarning: The following device_map keys do not match any submodules in the model: ['decoder.embed_tokens', 'encoder.embed_tokens']
  warnings.warn(


In [16]:
text_input = []
gloss_translate = []
answer = []

model.eval()

for batch in tqdm(test_dataloader):
    batch = {k: v.to(device) for k, v in batch.items()}

    with torch.no_grad():
        input_ids = batch["input_ids"]
        labels = batch["labels"]

        outputs = model.generate(
            input_ids=input_ids,
            attention_mask=batch["attention_mask"],
            max_new_tokens=300,
            length_penalty=0.6,
            early_stopping=True,
            num_beams=4,
            eos_token_id=tokenizer.eos_token_id,
            repetition_penalty=1.5,
        )

    input_ids = input_ids.cpu().numpy()
    outputs = outputs.cpu().numpy()
    labels = labels.cpu().numpy()

    # Replace -100 in labels with pad_token_id before decoding
    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)

    translation = tokenizer.batch_decode(outputs, skip_special_tokens=True)
    label = tokenizer.batch_decode(labels, skip_special_tokens=True)
    input = tokenizer.batch_decode(input_ids, skip_special_tokens=True)

    text_input.extend(input)
    gloss_translate.extend(translation)
    answer.extend(label)

result = pd.DataFrame({
    "text": text_input,
    "true_gloss": answer,
    "predicted_gloss": gloss_translate
})

  0%|          | 0/7 [00:00<?, ?it/s]

In [17]:
result

,text,true_gloss,predicted_gloss
0,thai to gloss: ชลบุรี ระยอง จันทบุรี และตราด,ชลบุรี|ระยอง|จันทบุรี|ตราด,ชลบุรี|ระยอง|จันทบุรี|ตราด
1,thai to gloss: ภาคกลางอุณหภูมิสูงขึ้น ประมาณ 1...,ภาคกลาง|*พื้นที่นี้(/ตรงนี้/ฝั่งนี้)|ร้อน|อุณห...,ภาคกลาง|ร้อน|อุณหภูมิสูง|#c 1-2[1 ถึง 2 ค้างท่...
2,thai to gloss: (คุณสุนิดา) สวัสดีค่ะคุณผู้ชม ม...,*ฉัน(/เรา)|สวัสดี|ดู|อากาศ|กับ|#s T|#s N|#s N|...,*ฉัน(/เรา)|สวัสดี|*วันนี้(/ช่วงนี้)|อากาศ|กับ|...
3,thai to gloss: ทะเลภาคตะวันออกวันนี้คลื่นต่ําก...,ภาคตะวันออก|*ทะเล(/คลื่น)|สูง|1|#s ม(เมตร)|ต่ํ...,ภาคตะวันออก|*ทะเล(/คลื่น)|สูง|1|#s ม(เมตร)|ต่ํ...
4,thai to gloss: กรมอุตุนิยมวิทยาคาดการณ์ 26-30 ...,#s ก|#s ร|#s ม|CL สถานที่|อากาศ|*ระบุ(/บอกว่า/...,กรม|CL สถานที่|อากาศ|*ระบุ(/บอกว่า/กล่าวไว้ว่า...
5,thai to gloss: แต่ว่าข้อดีก็มีนะคะ เพราะว่าฝนท...,ข้อดี|*ไหน(/อย่างไร)|สมมติ|*ฝนตกหนัก[ใช้สีหน้า...,#c ฝนตก + ในหลายพื้นที่[มีทิศทางประกอบตั้งแต่ฝ...
6,thai to gloss: ทีนี้เรามาตรวจสอบสภาพอากาศแบบรา...,*วันนี้(/ช่วงนี้)|บอกเล่า|อากาศ|*กับ(/ที่)|ภาค...,*วันนี้(/ช่วงนี้)|บอกเล่า|อากาศ|*กับ(/ที่)|ภาค...
7,thai to gloss: (คุณสุนิดา) สวัสดีค่ะคุณผู้ชม ม...,*ฉัน(/เรา)|สวัสดี|คน|ดู|*ทุกท่าน[การวนคือทุก ๆ...,*ฉัน(/เรา)|สวัสดี|*วันนี้(/ช่วงนี้)|อากาศ|กับ|...
8,thai to gloss: ส่วนภาคกลาง ช่วงเช้าเช้ายังมีอา...,*พื้นที่นี้(/ตรงนี้/ฝั่งนี้)|ภาคกลาง|ตอนเช้า|เ...,ภาคกลาง|ตอนเช้า|เย็น|สบาย|อุณหภูมิต่ํา|20|ร้อน...
9,thai to gloss: ปิดท้ายกันที่กรุงเทพมหานครและปร...,#c กรุงเทพมหานคร + จังหวัด + พื้นที่ใกล้เคียง(...,สุดท้าย|#c กรุงเทพมหานคร + จังหวัด + พื้นที่ใก...


In [18]:
result.to_csv("/project/lt200246-mmacma/Big_seq2seq/transcript/dataset_ver1/mt5/mt5_dataver1.csv")